# Health Check del entorno de AI

Este notebook valida que el entorno tenga lo necesario para comenzar a trabajar con ciencia de datos, Machine Learning, Deep Learning y Transformers.

Ejecuta **Run > Run All Cells**. Al final aparecerá un resumen con pruebas aprobadas y fallidas. No descarga modelos ni necesita Internet.

In [ ]:
import sys
import os
import platform
import importlib
import importlib.metadata as metadata
from pathlib import Path
from datetime import datetime

print(f"Fecha: {datetime.now().isoformat(timespec='seconds')}")
print(f"Python: {sys.version}")
print(f"Ejecutable: {sys.executable}")
print(f"Sistema: {platform.platform()}")
print(f"Arquitectura: {platform.machine()}")
print(f"CPU lógicas: {os.cpu_count()}")
print(f"Directorio actual: {Path.cwd()}")

## 1. Versiones instaladas

Se consulta la versión del paquete instalado y se prueba su importación. Una versión puede aparecer aunque el import falle por una dependencia binaria incompatible.

In [ ]:
packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "seaborn": "seaborn",
    "Pillow": "PIL",
    "plotly": "plotly",
    "torch": "torch",
    "torchvision": "torchvision",
    "torchaudio": "torchaudio",
    "transformers": "transformers",
    "datasets": "datasets",
    "accelerate": "accelerate",
    "evaluate": "evaluate",
    "tokenizers": "tokenizers",
    "sentencepiece": "sentencepiece",
    "safetensors": "safetensors",
    "huggingface-hub": "huggingface_hub",
    "tensorboard": "tensorboard",
    "opencv-python-headless": "cv2",
    "jupyterlab": "jupyterlab",
    "notebook": "notebook",
    "ipykernel": "ipykernel",
    "ipywidgets": "ipywidgets",
    "requests": "requests",
    "PyYAML": "yaml",
    "tqdm": "tqdm",
}

results = []
for distribution, module in packages.items():
    try:
        version = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        version = "NO INSTALADO"
    try:
        importlib.import_module(module)
        status = "OK"
        detail = ""
    except Exception as exc:
        status = "ERROR"
        detail = f"{type(exc).__name__}: {exc}"
    results.append((distribution, version, status, detail))

try:
    from IPython.display import display
    import pandas as pd
    display(pd.DataFrame(results, columns=["Paquete", "Versión", "Import", "Detalle"]))
except Exception:
    for row in results:
        print(row)

## 2. PyTorch y hardware

Comprueba el backend disponible y ejecuta multiplicación de matrices y diferenciación automática.

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"CUDA compilada en PyTorch: {torch.version.cuda}")
print(f"MPS disponible: {hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()}")
print(f"Threads CPU: {torch.get_num_threads()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
print(f"Dispositivo de prueba: {device}")

x = torch.randn(256, 256, device=device, requires_grad=True)
y = (x @ x.T).mean()
y.backward()
assert x.grad is not None and torch.isfinite(y), "Falló la operación de PyTorch"
print(f"Operación tensorial y autograd: OK (resultado={y.item():.6f})")

## 3. Ciencia de datos y Machine Learning

Valida NumPy, pandas y scikit-learn entrenando un modelo pequeño completamente en memoria.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
assert accuracy >= 0.80, f"Exactitud inesperadamente baja: {accuracy:.3f}"
print(f"NumPy: OK, forma={X.shape}")
print(f"pandas: OK, DataFrame={pd.DataFrame(X).shape}")
print(f"scikit-learn: OK, accuracy={accuracy:.3f}")

## 4. Visualización

Genera una gráfica sencilla para comprobar el backend de Matplotlib.

In [ ]:
import matplotlib.pyplot as plt

t = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, np.sin(t), label="sin(x)")
ax.plot(t, np.cos(t), label="cos(x)")
ax.set_title("Health check de Matplotlib")
ax.grid(alpha=0.3)
ax.legend()
plt.show()
print("Matplotlib: OK")

## 5. Ecosistema Hugging Face sin descargas

Crea una configuración y un tokenizador local de prueba. Esto verifica las librerías, pero no descarga pesos de modelos.

In [ ]:
from transformers import BertConfig, BertModel
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace

config = BertConfig(
    vocab_size=100, hidden_size=32, num_hidden_layers=1,
    num_attention_heads=4, intermediate_size=64
)
tiny_model = BertModel(config)
parameter_count = sum(p.numel() for p in tiny_model.parameters())

tokenizer = Tokenizer(WordLevel({"[UNK]": 0, "hola": 1, "ai": 2}, unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
tokens = tokenizer.encode("hola ai").tokens
assert tokens == ["hola", "ai"]
print(f"Transformers: OK, modelo local de prueba con {parameter_count:,} parámetros")
print(f"Tokenizers: OK, tokens={tokens}")

## 6. Escritura en el workspace

Comprueba que notebooks, datos, modelos y salidas sean accesibles desde el contenedor.

In [ ]:
workspace = Path("/workspace") if Path("/workspace").exists() else Path.cwd().parent
required_dirs = ["notebooks", "data", "models", "outputs", "src"]
write_results = []
for name in required_dirs:
    directory = workspace / name
    exists = directory.is_dir()
    writable = os.access(directory, os.W_OK) if exists else False
    write_results.append((str(directory), exists, writable))
    print(f"{directory}: existe={exists}, escritura={writable}")
assert all(exists and writable for _, exists, writable in write_results), "Hay directorios ausentes o sin permisos"

## 7. Resumen final

Vuelve a ejecutar los imports y presenta el estado general. Las pruebas funcionales anteriores también deben ejecutarse sin excepciones.

In [ ]:
failed = [(name, version, detail) for name, version, status, detail in results if status != "OK"]
print("=" * 70)
if failed:
    print(f"HEALTH CHECK: ATENCIÓN — {len(failed)} paquete(s) con problemas")
    for name, version, detail in failed:
        print(f"- {name} ({version}): {detail}")
else:
    print(f"HEALTH CHECK: OK — {len(results)} paquetes importados correctamente")
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | dispositivo {device}")
print("=" * 70)

assert not failed, "Revisa los paquetes marcados con ERROR arriba"